In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 66'S REAL POLICY
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 66's Real Policy")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB66_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_66_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB66_SUMMARY_PATH, "run 66_profitability_modeling_business_understanding.ipynb first (Problem 13)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB66_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB66_SUMMARY = json.load(f)

POLICY_PATH = Path(NB66_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(f"{POLICY_PATH} not found.\nFix: re-run Notebook 66.")
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    PROFITABILITY_MODELING_POLICY = json.load(f)

REAL_SPEND_COLUMNS = PROFITABILITY_MODELING_POLICY["real_spend_columns"]
REVENUE_ASSUMPTIONS = PROFITABILITY_MODELING_POLICY["revenue_assumptions"]
AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD = REVENUE_ASSUMPTIONS["avg_monthly_revenue_per_account_usd"]["value"]
REVENUE_MULTIPLIER_FLOOR = REVENUE_ASSUMPTIONS["revenue_multiplier_floor"]["value"]
REVENUE_MULTIPLIER_CEILING = REVENUE_ASSUMPTIONS["revenue_multiplier_ceiling"]["value"]
PROFITABILITY_TIER_NAMES = PROFITABILITY_MODELING_POLICY["profitability_tier_names"]
PROFITABILITY_TIER_CUT_PERCENTILES = PROFITABILITY_MODELING_POLICY["profitability_tier_cut_percentiles"]
KPI_TARGETS = PROFITABILITY_MODELING_POLICY["kpi_targets"]

# --- Real, already-validated sources this notebook reuses -- every value
#     below is read from Notebook 66's own recorded policy, never re-derived
#     or guessed. ---
P1_REUSE = PROFITABILITY_MODELING_POLICY["reused_from_problem_1"]
CHAMPION_NAME = P1_REUSE["champion_model"]

P8_REUSE = PROFITABILITY_MODELING_POLICY["reused_from_problem_8"]
EAD_PER_ACCOUNT_USD = P8_REUSE["ead_per_account_usd"]
LGD_ASSUMPTION = P8_REUSE["lgd_assumption"]

P12_REUSE = PROFITABILITY_MODELING_POLICY["reused_from_problem_12"]
P12_PROFILE_PATH = Path(P12_REUSE["profile_path"])
if not P12_PROFILE_PATH.exists():
    raise FileNotFoundError(f"{P12_PROFILE_PATH} not found.\nFix: re-run Notebook 63 (Problem 12).")
P12_RECOMMENDED_FOR_PRODUCTION = P12_REUSE["recommended_for_production"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]

if "profitability_modeling_modeling" in PILLAR_DIRS:
    PROFITABILITY_MODELING_DIR = PILLAR_DIRS["profitability_modeling_modeling"]
else:
    PROFITABILITY_MODELING_DIR = (
        PROJECT_ROOT / "Phase5_Customer_Business_Intelligence"
        / "13_Problem13_Risk_Adjusted_Profitability_Modeling" / "modeling"
    )
    print(f"NOTE: 'profitability_modeling_modeling' not in pillar_dirs -- using fallback: "
          f"{PROFITABILITY_MODELING_DIR}")
PROFITABILITY_MODELING_DIR.mkdir(parents=True, exist_ok=True)
PROFITABILITY_CHARTS_DIR = PROFITABILITY_MODELING_DIR / "charts"
PROFITABILITY_CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded Notebook 66's real policy from: {POLICY_PATH}")
print(f"Real Spend columns (Notebook 66)     : {len(REAL_SPEND_COLUMNS)}")
print(f"Reused Problem 1's champion (measured): {CHAMPION_NAME}")
print(f"Reused Problem 8's EAD/LGD (measured) : ${EAD_PER_ACCOUNT_USD:,} / {LGD_ASSUMPTION:.0%}")
print(f"Reused Problem 12's real unified profile: {P12_PROFILE_PATH}")
print(f"Modeling artifacts will be written under: {PROFITABILITY_MODELING_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 5 CAP,
#            WITH THE TWO-TIER RAM GUARD ESTABLISHED AFTER THE REAL
#            NOTEBOOK 52 FREEZE -- THIS NOTEBOOK DOES ONE REAL STREAMING
#            PASS OVER THE RAW CSV, THE SAME CLASS OF OPERATION)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE5_CPU_FRACTION_CAP = 0.92
_PHASE5_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(_historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE5_CPU_FRACTION_CAP)))
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE5_RAM_FRACTION_CAP))

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from scipy.stats import spearmanr
except ImportError:
    missing.append("scipy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs. Close other Jupyter kernels / "
        f"memory-heavy applications, confirm with `psutil.virtual_memory().available / 1e9`, then re-run "
        f"this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB available "
          f"(comfortable margin is {_comfortable_available_ram_gb:.2f} GB). Proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available >= "
          f"{_comfortable_available_ram_gb:.2f} GB comfortable margin.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling: {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"Raw train_data.csv                                          : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv (internal train, Notebook 02's real split)  : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD PROBLEM 12'S REAL UNIFIED PROFILE (BASE POPULATION)
# =============================================================================
_section("SECTION 4: Load Problem 12's Real Unified Profile (Base Population)")

P12_PROFILE_DF = pl.read_parquet(P12_PROFILE_PATH)
print(f"Loaded Problem 12's real unified profile: {P12_PROFILE_DF.height:,} customers, "
      f"columns: {P12_PROFILE_DF.columns}")
if "target" not in P12_PROFILE_DF.columns:
    raise RuntimeError("Problem 12's persisted profile is missing the real 'target' column -- cannot validate "
                        "this problem's hard-gating KPIs without it.")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: COMPUTE EACH CUSTOMER'S REAL SPEND_PERCENTILE_RANK (ONE REAL
#            STREAMING PASS OVER THE RAW CSV)
# =============================================================================
_section("SECTION 5: Compute Each Customer's Real SPEND_PERCENTILE_RANK")

# --- Real, honest scope decision: mirrors Notebook 63's "current status"
#     framing for DYNAMIC_PD/RISK_LEVEL -- this notebook uses each
#     customer's own real LATEST statement's average across the real Spend
#     columns Notebook 66 discovered, not a full-history average, so the
#     revenue signal reflects a customer's most current real spend behavior. ---
_spend_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in REAL_SPEND_COLUMNS:
    _spend_schema_overrides[_c] = pl.Float32

print(f"Streaming the real raw CSV once for each customer's real latest-statement average Spend signal "
      f"({RAW_TRAIN_DATA_PATH.stat().st_size / 1e9:.2f} GB) across {len(REAL_SPEND_COLUMNS)} real Spend "
      f"columns. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")
_t0 = time.time()
SPEND_DF = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_spend_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .group_by("customer_ID", maintain_order=False)
    .agg([pl.col(c).last().alias(c) for c in REAL_SPEND_COLUMNS])
    .with_columns(pl.mean_horizontal(REAL_SPEND_COLUMNS).alias("SPEND_RAW"))
    .select(["customer_ID", "SPEND_RAW"])
    .collect(engine="streaming")
)
print(f"Built in {time.time() - _t0:.1f}s: {SPEND_DF.height:,} customers' real latest-statement average "
      f"Spend signal. RSS: {_rss_gb():.2f} GB, available RAM: {_available_ram_gb():.2f} GB")

_n_null_spend = int(SPEND_DF["SPEND_RAW"].is_null().sum())
print(f"Real customers with an all-null Spend signal on their latest statement: {_n_null_spend:,} / "
      f"{SPEND_DF.height:,} -- these honestly fall back to the population median percentile rank (0.5) "
      f"below, never imputed with a fabricated raw value.")

_non_null_spend = SPEND_DF.filter(pl.col("SPEND_RAW").is_not_null())
_n_non_null = _non_null_spend.height
_ranked = _non_null_spend.with_columns(
    ((pl.col("SPEND_RAW").rank(method="average") - 1.0) / max(_n_non_null - 1, 1)).alias("SPEND_PERCENTILE_RANK")
).select(["customer_ID", "SPEND_PERCENTILE_RANK"])
SPEND_DF = (
    SPEND_DF.join(_ranked, on="customer_ID", how="left")
    .with_columns(pl.col("SPEND_PERCENTILE_RANK").fill_null(0.5))
    .select(["customer_ID", "SPEND_PERCENTILE_RANK"])
)
print(f"Real SPEND_PERCENTILE_RANK range: [{SPEND_DF['SPEND_PERCENTILE_RANK'].min():.4f}, "
      f"{SPEND_DF['SPEND_PERCENTILE_RANK'].max():.4f}], mean {SPEND_DF['SPEND_PERCENTILE_RANK'].mean():.4f} "
      f"(expected ~0.5 for a real, evenly-spread rank)")
del _non_null_spend, _ranked
gc.collect()
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: COMPUTE REAL REVENUE, PD-ADJUSTED REVENUE, EXPECTED LOSS &
#            PROFITABILITY_SCORE (NOTEBOOK 66'S FORMULA)
# =============================================================================
_section("SECTION 6: Compute Real Revenue, PD-Adjusted Revenue, Expected Loss & PROFITABILITY_SCORE")

PROFITABILITY_DF = P12_PROFILE_DF.join(SPEND_DF, on="customer_ID", how="left").with_columns(
    pl.col("SPEND_PERCENTILE_RANK").fill_null(0.5)
)
_n_unmatched = int(PROFITABILITY_DF["SPEND_PERCENTILE_RANK"].is_null().sum())
print(f"Customers in Problem 12's profile with no matching real Spend record: {_n_unmatched:,} "
      f"(should be 0 -- both are built from the same real raw CSV universe).")

PROFITABILITY_DF = PROFITABILITY_DF.with_columns([
    (REVENUE_MULTIPLIER_FLOOR + (REVENUE_MULTIPLIER_CEILING - REVENUE_MULTIPLIER_FLOOR)
     * pl.col("SPEND_PERCENTILE_RANK")).alias("REVENUE_MULTIPLIER"),
]).with_columns([
    (AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD * pl.col("REVENUE_MULTIPLIER")).alias("REVENUE_PER_ACCOUNT_USD"),
]).with_columns([
    (pl.col("REVENUE_PER_ACCOUNT_USD") * (1.0 - pl.col("UNIFIED_RISK_SCORE"))).alias("PD_ADJUSTED_REVENUE_USD"),
    (pl.col("UNIFIED_RISK_SCORE") * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION).alias("EXPECTED_LOSS_USD"),
]).with_columns([
    (pl.col("PD_ADJUSTED_REVENUE_USD") - pl.col("EXPECTED_LOSS_USD")).alias("PROFITABILITY_SCORE"),
])

print(f"Real REVENUE_PER_ACCOUNT_USD range : [${PROFITABILITY_DF['REVENUE_PER_ACCOUNT_USD'].min():,.2f}, "
      f"${PROFITABILITY_DF['REVENUE_PER_ACCOUNT_USD'].max():,.2f}], "
      f"mean ${PROFITABILITY_DF['REVENUE_PER_ACCOUNT_USD'].mean():,.2f}")
print(f"Real EXPECTED_LOSS_USD range       : [${PROFITABILITY_DF['EXPECTED_LOSS_USD'].min():,.2f}, "
      f"${PROFITABILITY_DF['EXPECTED_LOSS_USD'].max():,.2f}], "
      f"mean ${PROFITABILITY_DF['EXPECTED_LOSS_USD'].mean():,.2f}")
print(f"Real PROFITABILITY_SCORE range     : [${PROFITABILITY_DF['PROFITABILITY_SCORE'].min():,.2f}, "
      f"${PROFITABILITY_DF['PROFITABILITY_SCORE'].max():,.2f}], "
      f"mean ${PROFITABILITY_DF['PROFITABILITY_SCORE'].mean():,.2f}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: FIT REAL TERTILE CUTS (TRAIN SPLIT) & ASSIGN PROFITABILITY_TIER
# =============================================================================
_section("SECTION 7: Fit Real Tertile Cuts (TRAIN Split) & Assign PROFITABILITY_TIER")

TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TEST_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8}).select("customer_ID")
TRAIN_DF = PROFITABILITY_DF.join(TRAIN_IDS_DF, on="customer_ID", how="inner")
HOLDOUT_DF = PROFITABILITY_DF.join(TEST_IDS_DF, on="customer_ID", how="inner")
print(f"Real internal TRAIN split (fits tertile cuts): {TRAIN_DF.height:,} customers")
print(f"Real internal HOLDOUT split (validates KPIs) : {HOLDOUT_DF.height:,} customers")

_p_lo, _p_hi = PROFITABILITY_TIER_CUT_PERCENTILES[0] / 100.0, PROFITABILITY_TIER_CUT_PERCENTILES[1] / 100.0
PROFITABILITY_CUT_LOW = float(TRAIN_DF["PROFITABILITY_SCORE"].quantile(_p_lo))
PROFITABILITY_CUT_HIGH = float(TRAIN_DF["PROFITABILITY_SCORE"].quantile(_p_hi))
print(f"PROFITABILITY_TIER cuts (fit on TRAIN, {PROFITABILITY_TIER_CUT_PERCENTILES} percentiles): "
      f"low=${PROFITABILITY_CUT_LOW:,.2f}, high=${PROFITABILITY_CUT_HIGH:,.2f}")


def _assign_tier(df: "pl.DataFrame") -> "pl.DataFrame":
    _tier_expr = (
        pl.when(pl.col("PROFITABILITY_SCORE") <= PROFITABILITY_CUT_LOW).then(pl.lit(PROFITABILITY_TIER_NAMES[0]))
        .when(pl.col("PROFITABILITY_SCORE") <= PROFITABILITY_CUT_HIGH).then(pl.lit(PROFITABILITY_TIER_NAMES[1]))
        .otherwise(pl.lit(PROFITABILITY_TIER_NAMES[2]))
        .alias("PROFITABILITY_TIER")
    )
    return df.with_columns(_tier_expr)


PROFITABILITY_DF = _assign_tier(PROFITABILITY_DF)
TRAIN_DF = _assign_tier(TRAIN_DF)
HOLDOUT_DF = _assign_tier(HOLDOUT_DF)

_tier_counts = PROFITABILITY_DF.group_by("PROFITABILITY_TIER").agg(pl.len().alias("n")).sort("PROFITABILITY_TIER")
print("Real PROFITABILITY_TIER population counts:")
for _row in _tier_counts.iter_rows(named=True):
    print(f"  {_row['PROFITABILITY_TIER']:<20}: {_row['n']:>8,} "
          f"({100.0 * _row['n'] / PROFITABILITY_DF.height:.1f}%)")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: VALIDATE HARD-GATING KPIS ON THE REAL HOLDOUT SPLIT
# =============================================================================
_section("SECTION 8: Validate Hard-Gating KPIs on the Real HOLDOUT Split")

# --- profitability_tier_monotonicity: real observed default rate must be
#     non-increasing from Low to High Profitability tier. ---
_tier_default_rates = {}
for _tier in PROFITABILITY_TIER_NAMES:
    _tier_df = HOLDOUT_DF.filter(pl.col("PROFITABILITY_TIER") == _tier)
    _tier_default_rates[_tier] = float(_tier_df["target"].mean()) if _tier_df.height else float("nan")
print("Real HOLDOUT default rate by PROFITABILITY_TIER:")
for _tier in PROFITABILITY_TIER_NAMES:
    print(f"  {_tier:<20}: {_tier_default_rates[_tier]:.4%}")

_ordered_rates = [_tier_default_rates[t] for t in PROFITABILITY_TIER_NAMES]
PROFITABILITY_TIER_MONOTONICITY_PASSED = bool(
    _ordered_rates[0] >= _ordered_rates[1] >= _ordered_rates[2]
)
print(f"\nprofitability_tier_monotonicity ({' >= '.join(PROFITABILITY_TIER_NAMES)} real default rate): "
      f"{'PASS' if PROFITABILITY_TIER_MONOTONICITY_PASSED else 'FAIL'}")

# --- risk_adjustment_materiality: the real Spearman rank correlation between
#     UNIFIED_RISK_SCORE and PROFITABILITY_SCORE on the real HOLDOUT split
#     must be materially negative (<= the ASSUMPTION threshold). ---
_risk_arr = HOLDOUT_DF["UNIFIED_RISK_SCORE"].to_numpy()
_profit_arr = HOLDOUT_DF["PROFITABILITY_SCORE"].to_numpy()
RISK_PROFITABILITY_SPEARMAN_CORR, _spearman_pvalue = spearmanr(_risk_arr, _profit_arr)
RISK_PROFITABILITY_SPEARMAN_CORR = float(RISK_PROFITABILITY_SPEARMAN_CORR)
_spearman_threshold = KPI_TARGETS["risk_adjustment_materiality"]["spearman_threshold"]
RISK_ADJUSTMENT_MATERIALITY_PASSED = bool(RISK_PROFITABILITY_SPEARMAN_CORR <= _spearman_threshold)
print(f"\nReal Spearman correlation, UNIFIED_RISK_SCORE vs. PROFITABILITY_SCORE (real HOLDOUT, "
      f"n={len(_risk_arr):,}): {RISK_PROFITABILITY_SPEARMAN_CORR:.4f} (p={_spearman_pvalue:.2e})")
print(f"risk_adjustment_materiality: {RISK_PROFITABILITY_SPEARMAN_CORR:.4f} <= {_spearman_threshold} -- "
      f"{'PASS' if RISK_ADJUSTMENT_MATERIALITY_PASSED else 'FAIL'}")

_expected_share_pct = 100.0 / len(PROFITABILITY_TIER_NAMES)
_min_tier_pct = KPI_TARGETS["min_tier_population_pct"] / 100.0 * _expected_share_pct
_holdout_tier_counts = HOLDOUT_DF.group_by("PROFITABILITY_TIER").agg(pl.len().alias("n"))
_undersized = [(r["PROFITABILITY_TIER"], r["n"]) for r in _holdout_tier_counts.iter_rows(named=True)
               if 100.0 * r["n"] / HOLDOUT_DF.height < _min_tier_pct]
MIN_TIER_POPULATION_PASSED = len(_undersized) == 0
print(f"\nmin_tier_population_pct check: {'PASS' if MIN_TIER_POPULATION_PASSED else 'FAIL'} "
      f"(min real share required: {_min_tier_pct:.2f}% of HOLDOUT)")

ALL_HARD_GATES_PASSED = bool(PROFITABILITY_TIER_MONOTONICITY_PASSED and RISK_ADJUSTMENT_MATERIALITY_PASSED)
RECOMMENDED_FOR_PRODUCTION = bool(ALL_HARD_GATES_PASSED and MIN_TIER_POPULATION_PASSED)
if not ALL_HARD_GATES_PASSED:
    print(
        "\nHONEST FINDING: one or more hard-gating KPIs did not pass on this real run. This notebook still "
        "proceeds to report the full population (Sections 9-11 below) for completeness -- the same honest "
        "'not yet viable' standard Notebooks 36/40/48/56 held their own problems to -- but every downstream "
        "artifact and report for Problem 13 marks this run NOT RECOMMENDED FOR PRODUCTION."
    )
print(f"\nRECOMMENDED_FOR_PRODUCTION (this run): {RECOMMENDED_FOR_PRODUCTION}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: REAL PER-TIER P&L SUMMARY (STANDING METRICS-DISPLAY RULE)
# =============================================================================
_section("SECTION 9: Real Per-Tier P&L Summary")

TIER_PL_SUMMARY = []
for _tier in PROFITABILITY_TIER_NAMES:
    _tdf = HOLDOUT_DF.filter(pl.col("PROFITABILITY_TIER") == _tier)
    TIER_PL_SUMMARY.append({
        "tier": _tier, "n": _tdf.height,
        "mean_unified_risk_score": float(_tdf["UNIFIED_RISK_SCORE"].mean()) if _tdf.height else None,
        "mean_revenue_per_account_usd": float(_tdf["REVENUE_PER_ACCOUNT_USD"].mean()) if _tdf.height else None,
        "mean_pd_adjusted_revenue_usd": float(_tdf["PD_ADJUSTED_REVENUE_USD"].mean()) if _tdf.height else None,
        "mean_expected_loss_usd": float(_tdf["EXPECTED_LOSS_USD"].mean()) if _tdf.height else None,
        "mean_profitability_score_usd": float(_tdf["PROFITABILITY_SCORE"].mean()) if _tdf.height else None,
        "real_default_rate": _tier_default_rates[_tier],
    })
print(f"{'Tier':<20}{'N':>10}{'Mean Risk':>12}{'Mean Rev':>12}{'Mean PD-Adj Rev':>18}"
      f"{'Mean Exp Loss':>16}{'Mean Profit':>14}{'Default Rate':>14}")
for _row in TIER_PL_SUMMARY:
    print(f"{_row['tier']:<20}{_row['n']:>10,}{_row['mean_unified_risk_score']:>12.4f}"
          f"${_row['mean_revenue_per_account_usd']:>10,.2f}${_row['mean_pd_adjusted_revenue_usd']:>16,.2f}"
          f"${_row['mean_expected_loss_usd']:>14,.2f}${_row['mean_profitability_score_usd']:>12,.2f}"
          f"{_row['real_default_rate']:>13.2%}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: CHARTS
# =============================================================================
_section("SECTION 10: Charts")

plt.figure(figsize=(7.5, 5))
_bar_x = [r["tier"] for r in TIER_PL_SUMMARY]
_bar_profit = [r["mean_profitability_score_usd"] for r in TIER_PL_SUMMARY]
_line_default = [r["real_default_rate"] * 100 for r in TIER_PL_SUMMARY]
_fig, _ax1 = plt.subplots(figsize=(7.5, 5))
_ax1.bar(_bar_x, _bar_profit, color=["#e76f51", "#e9c46a", "#2a9d8f"])
_ax1.set_ylabel("Mean PROFITABILITY_SCORE (USD)")
_ax2 = _ax1.twinx()
_ax2.plot(_bar_x, _line_default, color="#0B1F3A", marker="o", linewidth=2, label="Real Default Rate")
_ax2.set_ylabel("Real Default Rate (%)")
plt.title("Problem 13 -- Real Mean Profitability & Default Rate by Tier (Real Holdout)")
plt.tight_layout()
_tier_chart_path = PROFITABILITY_CHARTS_DIR / "profitability_tier_pl_chart.png"
plt.savefig(_tier_chart_path, dpi=120)
plt.close()

plt.figure(figsize=(7, 6))
plt.hexbin(_risk_arr, _profit_arr, gridsize=40, cmap="viridis", mincnt=1)
plt.colorbar(label="Real Customer Count")
plt.xlabel("UNIFIED_RISK_SCORE (Problem 12, real)")
plt.ylabel("PROFITABILITY_SCORE (USD, real)")
plt.title(f"Problem 13 -- Risk vs. Profitability (Real Holdout, Spearman={RISK_PROFITABILITY_SPEARMAN_CORR:.3f})")
plt.tight_layout()
_scatter_chart_path = PROFITABILITY_CHARTS_DIR / "risk_vs_profitability_hexbin_chart.png"
plt.savefig(_scatter_chart_path, dpi=120)
plt.close()

print(f"Wrote: {_tier_chart_path}")
print(f"Wrote: {_scatter_chart_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: PERSIST THE REAL PROFITABILITY-SCORED PROFILE & MODELING
#             RESULTS
# =============================================================================
_section("SECTION 11: Persist the Real Profitability-Scored Profile & Modeling Results")

_profile_cols = ["customer_ID", "UNIFIED_RISK_SCORE", "UNIFIED_RISK_GRADE", "SPEND_PERCENTILE_RANK",
                  "REVENUE_MULTIPLIER", "REVENUE_PER_ACCOUNT_USD", "PD_ADJUSTED_REVENUE_USD",
                  "EXPECTED_LOSS_USD", "PROFITABILITY_SCORE", "PROFITABILITY_TIER", "target"]
PROFITABILITY_PROFILE = PROFITABILITY_DF.select(_profile_cols).sort("PROFITABILITY_SCORE", descending=True)
profile_path = PROFITABILITY_MODELING_DIR / "profitability_scored_profile.parquet"
PROFITABILITY_PROFILE.write_parquet(profile_path)
print(f"Wrote: {profile_path} ({profile_path.stat().st_size / 1e6:.1f} MB, "
      f"{PROFITABILITY_PROFILE.height:,} customers)")

MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 13 -- Risk-Adjusted Profitability Modeling (Modeling)",
    "eligible_population": PROFITABILITY_DF.height,
    "train_split_population": TRAIN_DF.height,
    "holdout_split_population": HOLDOUT_DF.height,
    "profitability_cut_low_usd": PROFITABILITY_CUT_LOW,
    "profitability_cut_high_usd": PROFITABILITY_CUT_HIGH,
    "kpi_results": {
        "profitability_tier_monotonicity": {
            "real_default_rate_by_tier": _tier_default_rates, "passed": PROFITABILITY_TIER_MONOTONICITY_PASSED,
        },
        "risk_adjustment_materiality": {
            "spearman_correlation": RISK_PROFITABILITY_SPEARMAN_CORR, "spearman_pvalue": float(_spearman_pvalue),
            "threshold": _spearman_threshold, "passed": RISK_ADJUSTMENT_MATERIALITY_PASSED,
        },
        "min_tier_population_pct": {"passed": MIN_TIER_POPULATION_PASSED,
                                     "undersized_tiers": [{"tier": t, "n": n} for t, n in _undersized]},
    },
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "tier_pl_summary": TIER_PL_SUMMARY,
    "profile_path": str(profile_path),
    "random_seed": RANDOM_SEED,
}
modeling_results_path = PROFITABILITY_MODELING_DIR / "profitability_modeling_results.json"
with open(modeling_results_path, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS, f, indent=2)
print(f"Wrote: {modeling_results_path}")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 12: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Profitability-scored profile file was written", profile_path.exists())
_all_checks_passed &= _check("Modeling results file was written", modeling_results_path.exists())
_all_checks_passed &= _check("PROFITABILITY_SCORE is finite and non-null for every customer",
                              int(PROFITABILITY_DF["PROFITABILITY_SCORE"].is_null().sum()) == 0
                              and bool(PROFITABILITY_DF["PROFITABILITY_SCORE"].is_finite().all()))
_all_checks_passed &= _check("Every customer was assigned a real PROFITABILITY_TIER",
                              PROFITABILITY_DF["PROFITABILITY_TIER"].is_in(PROFITABILITY_TIER_NAMES).all())
_all_checks_passed &= _check("SPEND_PERCENTILE_RANK is bounded in [0, 1] for every customer",
                              bool((PROFITABILITY_DF["SPEND_PERCENTILE_RANK"] >= 0.0).all())
                              and bool((PROFITABILITY_DF["SPEND_PERCENTILE_RANK"] <= 1.0).all()))
_all_checks_passed &= _check("Profile row count matches the eligible population count",
                              PROFITABILITY_PROFILE.height == PROFITABILITY_DF.height)
_all_checks_passed &= _check("PROFITABILITY_TIER cut values are correctly ordered (low < high)",
                              PROFITABILITY_CUT_LOW < PROFITABILITY_CUT_HIGH)
_all_checks_passed &= _check("EAD/LGD were inherited from Notebook 66's policy (ultimately Notebook 08), "
                              "not re-guessed",
                              EAD_PER_ACCOUNT_USD == P8_REUSE["ead_per_account_usd"]
                              and LGD_ASSUMPTION == P8_REUSE["lgd_assumption"])
_all_checks_passed &= _check("Train and holdout splits do not overlap",
                              len(set(TRAIN_DF["customer_ID"]) & set(HOLDOUT_DF["customer_ID"])) == 0)
_all_checks_passed &= _check("Spearman correlation is a valid value in [-1, 1]",
                              -1.0 <= RISK_PROFITABILITY_SPEARMAN_CORR <= 1.0)
_all_checks_passed &= _check("KPI results object reused the exact same real correlation computed above "
                              "(no re-derivation)",
                              MODELING_RESULTS["kpi_results"]["risk_adjustment_materiality"]["spearman_correlation"]
                              == RISK_PROFITABILITY_SPEARMAN_CORR)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 12 complete -- all checks passed.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 67 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 13: Write Notebook 67 Summary Artifact")

NB67_SUMMARY = {
    "notebook": "67_profitability_modeling_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(modeling_results_path),
    "profile_path": str(profile_path),
    "eligible_population": PROFITABILITY_DF.height,
    "profitability_cut_low_usd": PROFITABILITY_CUT_LOW,
    "profitability_cut_high_usd": PROFITABILITY_CUT_HIGH,
    "all_hard_gates_passed": ALL_HARD_GATES_PASSED,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "risk_profitability_spearman_corr": RISK_PROFITABILITY_SPEARMAN_CORR,
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB67_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_67_summary.json"
with open(NB67_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB67_SUMMARY, f, indent=2)
print(f"Wrote: {NB67_SUMMARY_PATH}")

_section("NOTEBOOK 67 COMPLETE")
print(f"Real scored population (real)                  : {PROFITABILITY_DF.height:,} customers")
print(f"profitability_tier_monotonicity                : "
      f"{'PASS' if PROFITABILITY_TIER_MONOTONICITY_PASSED else 'FAIL'}")
print(f"risk_adjustment_materiality (Spearman={RISK_PROFITABILITY_SPEARMAN_CORR:.4f}) : "
      f"{'PASS' if RISK_ADJUSTMENT_MATERIALITY_PASSED else 'FAIL'}")
print(f"RECOMMENDED_FOR_PRODUCTION (this run)          : {RECOMMENDED_FOR_PRODUCTION}")
print(f"Profitability-scored profile written to: {profile_path}")
print(
    "\nNext: 68_profitability_modeling_validation_deployment.ipynb -- independently reproduces this notebook's "
    "real pipeline from scratch, bootstraps a confidence interval on the risk_adjustment_materiality Spearman "
    "correlation, verifies the persisted profile against a fresh reproduction, and packages the real FastAPI "
    "profitability-scoring lookup service."
)